# btp-offload-cache: selective offloading and allocator experiments

This separate notebook preserves baseline+ and btp-eval-optimised. It tests **which denoisers to offload** and **when to flush the CUDA allocator**, with identical weights, float32, seed, guidance and image geometry.

Use GPU T4 x2 and Internet. Workers use one T4 sequentially. Attach private datasets `btp-offload-cache-source`, `btp-paper-weights-2`, and `btp-eval-nh-haze`.

Default: six M0 repeatability jobs, then eight placement-screening jobs (14 jobs total). Based only on the previous M0 pilot, budget roughly **45 minutes of process runtime plus setup** if candidates cost similarly; actual costs and OOMs may differ. Per-job timeout: 8 minutes. Set `RUN_PLACEMENT=False` for the initial six-job baseline session. Cache and confirmation stages are opt-in after manual inspection. No policy is automatically declared a winner.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

subprocess.run(["nvidia-smi"], check=True)
# Attached private weights dataset; update the version if the dataset changes.
WEIGHTS_DATASET_VERSION = "cubingultimate/btp-paper-weights-2/versions/1"
assert not WEIGHTS_DATASET_VERSION.startswith("EDIT_ME"), (
    "Set the dataset version above first"
)
ROOT = Path("/kaggle/working/BTP")
OUTPUT = Path("/kaggle/working/btp-offload-cache-results") / (
    "session-" + str(time.time_ns())
)
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_PLACEMENT = True
RUN_CACHE = False
RUN_CONFIRMATION = False
# Choose after inspecting the screening report. M0 is a valid fallback.
CACHE_PLACEMENT = "M0"
CONFIRM_CACHE = "never"

## 1. Restore and verify the uploaded source bundle
Kaggle may keep the ZIP or expand it; both layouts are supported. Existing working copies are checked rather than silently overwritten.

In [ ]:
archives = list(Path("/kaggle/input").rglob("btp-offload-cache-source.bundle"))
if not archives:
    archives = list(Path("/kaggle/input").rglob("btp-offload-cache-source.zip"))
if ROOT.exists():
    shutil.move(str(ROOT), str(ROOT.with_name("BTP-backup-" + str(time.time_ns()))))
if not ROOT.exists():
    if len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.namelist():
                destination = (Path("/kaggle/working") / item).resolve()
                if not destination.is_relative_to(Path("/kaggle/working")):
                    raise ValueError("Unsafe archive member")
            archive.extractall("/kaggle/working")
    else:
        manifests = list(Path("/kaggle/input").rglob("source-manifest.json"))
        assert len(manifests) == 1, "Attach exactly one BTP source dataset"
        shutil.copytree(manifests[0].parent, ROOT)
snapshot = json.loads((ROOT / "source-manifest.json").read_text())
# Recover the exact upstream gzip if a dataset upload expanded it to .txt.
vocab_relative = "upstream/Learning-Hazing-to-Dehazing/diffbir/model/open_clip/bpe_simple_vocab_16e6.txt.gz"
vocab = ROOT / vocab_relative
if not vocab.exists() and vocab.with_suffix("").exists():
    import gzip
    import urllib.request

    commit = snapshot["upstream_commit"]
    assert len(commit) == 40 and all(c in "0123456789abcdef" for c in commit)
    url = (
        "https://raw.githubusercontent.com/ruiyi-w/Learning-Hazing-to-Dehazing/"
        + commit
        + "/diffbir/model/open_clip/bpe_simple_vocab_16e6.txt.gz"
    )
    with urllib.request.urlopen(url, timeout=60) as response:
        original_gzip = response.read()
    assert (
        hashlib.sha256(original_gzip).hexdigest() == snapshot["files"][vocab_relative]
    ), "Tokenizer download hash mismatch"
    assert gzip.decompress(original_gzip) == vocab.with_suffix("").read_bytes(), (
        "Expanded tokenizer differs from upstream"
    )
    vocab.write_bytes(original_gzip)
    print("Recovered verified tokenizer gzip:", vocab)

for relative, expected in snapshot["files"].items():
    # Finder metadata is not source and can be omitted during dataset upload.
    if Path(relative).name == ".DS_Store":
        continue
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, (
        relative
    )
print("Verified source snapshot:", snapshot["upstream_commit"])

## 2. Create an isolated Python 3.10 CUDA runtime
This downloads Python, CUDA-enabled PyTorch and dependencies. The environment is in `/tmp` so it is not included in saved notebook outputs. It leaves Kaggle's notebook-kernel packages intact.

In [ ]:
uv_binary = shutil.which("uv")
if uv_binary is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
    UV = [sys.executable, "-m", "uv"]
else:
    UV = [uv_binary]
ENV = Path("/tmp/btp-kaggle-env")
PYTHON = ENV / "bin/python"
if not PYTHON.exists():
    subprocess.run(UV + ["venv", "--python", "3.10", str(ENV)], check=True)
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "torch==2.6.0",
        "torchvision==0.21.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
    ],
    check=True,
)
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "-r",
        str(ROOT / "configs/kaggle-requirements.txt"),
    ],
    check=True,
)
RUN = UV + ["run", "--no-project", "--python", str(PYTHON), "python"]
ENVIRONMENT = os.environ.copy()
ENVIRONMENT.update(
    PYTHONPATH=str(ROOT / "src"), CUDA_VISIBLE_DEVICES="0", ATTN_MODE="sdp"
)
subprocess.run(
    RUN
    + [
        "-c",
        "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0)); print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory/1e9)",
    ],
    check=True,
    env=ENVIRONMENT,
)
with (OUTPUT / "runtime-packages.txt").open("w") as file:
    subprocess.run(
        UV + ["pip", "freeze", "--python", str(PYTHON)], check=True, stdout=file
    )

# Tiny CPU check in the existing inference environment; no extra packages.
subprocess.run(
    RUN + [str(ROOT / "scripts/smoke_memory.py")], check=True, env=ENVIRONMENT, cwd=ROOT
)

## 3. Locate and hash the three checkpoints
Weights remain on the read-only input mount. The two supplied Baidu files are byte-identical to our previous copies; this step detects incomplete/wrong uploads.

In [ ]:
catalog = json.loads((ROOT / "configs/checkpoints.json").read_text())
paths = {}
for name, spec in catalog.items():
    candidates = list(Path("/kaggle/input").rglob(name))
    assert len(candidates) == 1, f"Expected exactly one {name}, found {len(candidates)}"
    path = candidates[0]
    assert path.stat().st_size == spec["bytes"], f"Incomplete upload: {name}"
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(block)
    assert digest.hexdigest() == spec["sha256"], f"Hash mismatch: {name}"
    paths[name] = path
    print("Verified", name, digest.hexdigest())
# Files can be in different dataset directories; symlink them into a common folder.
WEIGHTS = Path("/tmp/btp-reference-weights")
WEIGHTS.mkdir(exist_ok=True)
for name, path in paths.items():
    target = WEIGHTS / name
    if target.is_symlink():
        assert target.resolve() == path.resolve(), (
            "Restart session after changing datasets"
        )
    elif target.exists():
        raise FileExistsError(target)
    else:
        target.symlink_to(path)
(OUTPUT / "weights-verification.json").write_text(
    json.dumps(
        {
            "dataset_version": WEIGHTS_DATASET_VERSION,
            "files": {
                name: {"path": str(path), **catalog[name]}
                for name, path in paths.items()
            },
        },
        indent=2,
    )
)

## 4. Tiny CPU smoke first
Random tiny networks test plumbing only. Their images are not pretrained results. This cell does not run training or a large model.

In [ ]:
subprocess.run(RUN + ["-m", "btp.model_smoke"], cwd=ROOT, env=ENVIRONMENT, check=True)

## 5. Restore the frozen paired dataset
The manifest verifies every image hash and pair, including held-out scenes. Only the two preselected development scenes enter the pilot. Source hashes, crop coordinates and split rules are available for inspection.


In [ ]:
archives = list(Path("/kaggle/input").rglob("nh-haze-512.bundle"))
assert len(archives) == 1, "Attach cubingultimate/btp-eval-nh-haze"
DATA = Path("/kaggle/working/nh-haze-eval-" + str(time.time_ns()))
DATA.mkdir()
with zipfile.ZipFile(archives[0]) as archive:
    for item in archive.namelist():
        assert (DATA / item).resolve().is_relative_to(DATA.resolve()), (
            "Unsafe dataset path"
        )
    archive.extractall(DATA)
MANIFEST = DATA / "nh_haze_manifest.json"
assert (
    MANIFEST.read_bytes()
    == (ROOT / "configs/evaluation/nh_haze_manifest.json").read_bytes()
), "Evaluation dataset version mismatch"
subprocess.run(
    RUN + ["-m", "btp.evaluation", "--manifest", str(MANIFEST)],
    cwd=ROOT,
    env=ENVIRONMENT,
    check=True,
)
provenance = {
    "notebook": "cubingultimate/btp-offload-cache",
    "source_dataset": "cubingultimate/btp-offload-cache-source",
    "evaluation_dataset": "cubingultimate/btp-eval-nh-haze/versions/1",
    "evaluation_archive_sha256": hashlib.sha256(archives[0].read_bytes()).hexdigest(),
    "source_manifest_sha256": hashlib.sha256(
        (ROOT / "source-manifest.json").read_bytes()
    ).hexdigest(),
    "manifest_sha256": hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
    "baseline_template_sha256": "1ef0c9a1bb461698bdbd0459a3655f8715784e84c0a39761f57889330bce8522",
}
(OUTPUT / "notebook-provenance.json").write_text(json.dumps(provenance, indent=2))
with (OUTPUT / "nvidia-smi.txt").open("w") as file:
    subprocess.run(["nvidia-smi"], stdout=file, check=True)
print(json.dumps(provenance, indent=2))

## 6. Stage helpers and immutable evidence
Every stage gets a new directory. Results are archived after each subprocess, including failures. Jobs record both placement and cache mode; summary keys use the variant ID. `guidance` flushes after conditioning and after each actual offload; `phase` only after conditioning; `never` disables adapter-added flushing. All model/guidance computation is unchanged.


In [ ]:
def save_evidence():
    archive = shutil.make_archive(str(OUTPUT), "zip", OUTPUT)
    print("Evidence archive:", archive)
    return archive


def run_stage(name, config):
    config_file = OUTPUT / (name + "-config-" + str(time.time_ns()) + ".json")
    config_file.write_text(json.dumps(config, indent=2))
    jobs = (
        len(config["scene_ids"])
        * len(config["variants"])
        * len(config["seeds"])
        * len(config["modes"])
        * config["repeats"]
    )
    assert jobs <= config["max_jobs"]
    print(name, "jobs:", jobs, "worker timeout:", config["timeout_seconds"], "seconds")
    destination = OUTPUT / (name + "-" + str(time.time_ns()))
    try:
        subprocess.run(
            RUN
            + [
                "-u",
                "-m",
                "btp.benchmark",
                "--manifest",
                str(MANIFEST),
                "--config",
                str(config_file),
                "--weights",
                str(WEIGHTS),
                "--output",
                str(destination),
            ],
            cwd=ROOT,
            env=ENVIRONMENT,
            check=True,
        )
    finally:
        save_evidence()
    return destination


def load_config(name):
    return json.loads((ROOT / "configs/offload-cache" / (name + ".json")).read_text())


def analyze_stage(directory, baseline=None):
    report = directory / ("eligibility.json" if baseline else "repeatability.json")
    command = RUN + [
        "-m",
        "btp.optimization",
        "--run",
        str(directory),
        "--output",
        str(report),
    ]
    if baseline is not None:
        command += ["--baseline", str(baseline)]
    try:
        result = subprocess.run(command, cwd=ROOT, env=ENVIRONMENT, check=False)
        if result.returncode:
            print(
                "Evidence gate did not pass. Inspect logs; later stages remain disabled."
            )
            return None
        return json.loads(report.read_text())
    finally:
        save_evidence()

## 7. Measure M0 repeatability first (6 jobs)
Two fixed development scenes, seed 42, three repeats each. All jobs must complete without profiling. The report measures saved-PNG differences and timing variation. Observed variation is a conservative development check, not a universal floating-point tolerance or a test of latent-gradient equivalence.


In [ ]:
BASELINE = run_stage("repeatability", load_config("repeatability"))
REPEATABILITY = analyze_stage(BASELINE)
if REPEATABILITY is not None:
    print("Observed M0 MAE limit:", REPEATABILITY["mae"])
    print("Observed M0 maximum pixel difference:", REPEATABILITY["max_abs_error"])

## 8. Selective placement screening (8 jobs)
Compare M0 with UNet-only offload (M2-U), ControlNet-only offload (M2-C), and both denoisers resident (M2-R). Each uses `guidance` cache mode. OOMs remain in the table and make a variant ineligible. Screening checks matched outputs against M0 and the observed repeat variation; it never selects using clean-target scores. One repeat is screening only.


In [ ]:
SCREEN = None
ELIGIBILITY = None
if RUN_PLACEMENT and REPEATABILITY is not None:
    SCREEN = run_stage("placement", load_config("placement"))
    ELIGIBILITY = analyze_stage(SCREEN, BASELINE)
else:
    print("Placement skipped: disabled or baseline repeatability incomplete.")

## 9. Inspect and choose a placement manually
Use complete coverage, output differences, sampling latency and memory. `eligible_for_cache_test` only permits further testing; it does not establish speedup, measured total-device headroom or robustness on larger images. If a candidate is ineligible, investigate it; do not loosen the limits to make it pass. Keep M0 if no alternative is viable.


In [ ]:
import pandas as pd

if SCREEN is not None:
    display(pd.read_csv(SCREEN / "per_image.csv"))
if ELIGIBILITY is not None:
    display(pd.DataFrame.from_dict(ELIGIBILITY["variants"], orient="index"))
    print("Select CACHE_PLACEMENT from an eligible policy, then enable RUN_CACHE.")

## 10. Independent cache comparison on the selected placement (6 jobs, opt-in)
Change only cache mode: `guidance`, `phase`, `never`. The control is the selected placement with guidance flushing, not a different denoiser placement. Reducing cache release can affect OOMs and reservation; it does not reduce live tensors. Inspect all failures and numerical differences before choosing `CONFIRM_CACHE`.


In [ ]:
CACHE_RUN = None
if RUN_CACHE:
    assert ELIGIBILITY is not None, "Complete baseline and placement stages first"
    selected_id = CACHE_PLACEMENT + "-guidance"
    assert (
        ELIGIBILITY["variants"].get(selected_id, {}).get("eligible_for_cache_test")
    ), "Selected placement is not eligible"
    config = load_config("cache-template")
    config["variants"] = [
        {
            "id": CACHE_PLACEMENT + "-" + cache,
            "policy": CACHE_PLACEMENT,
            "cache_policy": cache,
        }
        for cache in ("guidance", "phase", "never")
    ]
    config["reference_variant"] = selected_id
    CACHE_RUN = run_stage("cache", config)
    display(pd.read_csv(CACHE_RUN / "per_image.csv"))
    print((CACHE_RUN / "summary.json").read_text())
else:
    print("Cache comparison is opt-in after selecting a placement.")

## 11. Confirm the chosen combination against M0 (12 jobs, opt-in)
Three repeats per scene/policy. The final comparator is M0 with its original guidance flushing. The 10% sampling-time reduction target is a project goal, not a promised gain. Inspect completion, numerical drift, per-image regressions, host RAM and memory. No automatic acceptance or held-out tuning occurs.


In [ ]:
CONFIRMATION = None
if RUN_CONFIRMATION:
    assert CACHE_RUN is not None and REPEATABILITY is not None, (
        "Run cache screening first"
    )
    chosen = CACHE_PLACEMENT + "-" + CONFIRM_CACHE
    cache_summary = json.loads((CACHE_RUN / "summary.json").read_text())
    info = cache_summary["policies"].get(chosen + "/accsamp")
    assert info and info["failed"] == 0 and info["completed"] == 2, (
        "Candidate cache mode did not complete both scenes"
    )
    comparison = cache_summary["comparisons"].get(chosen + "/accsamp")
    if comparison is not None:
        assert comparison["common_success_pairs"] == 2, "Cache control is incomplete"
        assert all(
            d["mae"] <= REPEATABILITY["mae"] + 1e-12
            and d["max_abs_error"] <= REPEATABILITY["max_abs_error"] + 1e-12
            for d in comparison["output_differences"]
        ), "Investigate cache output drift before confirmation"
    assert chosen != "M0-guidance", "M0-guidance is already the control"
    config = load_config("confirmation-template")
    config["variants"][1] = {
        "id": chosen,
        "policy": CACHE_PLACEMENT,
        "cache_policy": CONFIRM_CACHE,
    }
    CONFIRMATION = run_stage("confirmation", config)
    print(json.dumps(analyze_stage(CONFIRMATION, BASELINE), indent=2))
    display(pd.read_csv(CONFIRMATION / "per_image.csv"))
else:
    print("Confirmation is opt-in after cache screening.")

## 12. Download evidence
Save a Kaggle version with outputs and download the ZIP. Source/data hashes, package versions, nvidia-smi output, configs, failures, predictions, timing, memory and comparison reports are retained. Existing notebooks and their source datasets remain unchanged. These are development crops, not official paper benchmark results. LPIPS, model reuse, mixed precision and asynchronous transfers are outside this experiment.


In [ ]:
display(FileLink(save_evidence()))
print(
    "No optimization is accepted automatically. Review measured results and repeatability before selecting a default."
)